In [1]:
import cv2
import numpy as np

def preprocess(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    
    return blur


def segment(blur):
    _, thresh = cv2.threshold(blur, 120, 255, cv2.THRESH_BINARY_INV)
    return thresh


def get_largest_contour(thresh):
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    print("Contours found:", len(contours))

    if len(contours) == 0:
        return None

    contours = [c for c in contours if cv2.contourArea(c) > 1000]

    if len(contours) == 0:
        return None

    return max(contours, key=cv2.contourArea)

def measure_rotated(contour):
    rect = cv2.minAreaRect(contour)
    (cx, cy), (w, h), angle = rect
    
    if h > w:
        w, h = h, w
    
    return w, h, rect


def process_image(image_path, visualize=False):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Image not found")

    blur = preprocess(image)
    thresh = segment(blur)
    contour = get_largest_contour(thresh)

    if contour is None:
        print("No object detected")
        return None

    w, h, rect = measure_rotated(contour)    

    result = {
        "width": float(w),
        "height": float(h)
    }

    if visualize:
        vis = image.copy()

        box = cv2.boxPoints(rect)
        box = np.int32(box)
        cv2.drawContours(vis, [box], 0, (0, 255, 0), 2)

        # Put text
        text = f"{w:.1f} x {h:.1f}"
        cv2.putText(vis, text, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 255, 0), 2)

        cv2.imshow("Result", vis)
        cv2.imshow("Threshold", thresh)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    return result


if __name__ == "__main__":
    result = process_image(r"C:\Users\ASUS\Downloads\istockphoto-184276818-612x612.jpg")
    if result:
        print(f"Width: {result['width']:.2f}")
        print(f"Height: {result['height']:.2f}")

Contours found: 148
Width: 484.73
Height: 445.62


width:  AI      | Real    | Perbandingan
1:
Width:  1260.70 | 17cm    | 17.8cm
Height: 508.89  | 7.5cm   | 7.22cm
2:
Width:  513.00  | 6.4cm   | 7.25cm
Height: 511.00  | 6.4cm   | 7.26cm
3: Patokan
Width:  743.43  | 10.5cm  | 10.5cm
Height: 598.45  | 8.5cm   | 8.5cm